# <center>**Лабораторная работа №8** </center>

In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

### **1. Загрузите данные из файла data-logistic.csv.**

Это двумерная выборка, целевая переменная на которой принимает значения -1 или 1.

In [4]:
data = pd.read_csv('data-logistic.csv', header=None)

y = data.iloc[:, 0].values  
X = data.iloc[:, 1:].values  

### **2. Убедитесь, что выше выписаны правильные формулы для градиентного спуска.**

### **3. Реализуйте градиентный спуск для обычной и L2-регуляризованной (с коэффициентом регуляризации 10) логистической регрессии.**

Используйте длину шага k=0.1. В качестве начального приближения используйте вектор (0, 0).

In [16]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def gradient_descent(X, y, C=0, k=0.1, max_iter=10000, eps=1e-5):
    n_samples, n_features = X.shape
    w = np.zeros(n_features)
    
    for i in range(max_iter):
        linear = np.dot(X, w)
        sig = sigmoid(y * linear)
        
        grad = (1/n_samples) * np.sum(y * X.T * (1 - sig), axis=1)
        if C > 0:
            grad -= C * w
        
        w_new = w + k * grad
        
        if np.linalg.norm(w_new - w) < eps:
            return w_new, i + 1
        w = w_new
    
    return w, max_iter

w_no_reg, _ = gradient_descent(X, y, C=0, k=0.1)
w_reg, _ = gradient_descent(X, y, C=10, k=0.1)

print(f"Веса без регуляризации: w = {w_no_reg}")
print(f"Веса с регуляризацией: w = {w_reg}")

Веса без регуляризации: w = [0.28781162 0.0919833 ]
Веса с регуляризацией: w = [0.02855875 0.02478014]


### **4. Запустите градиентный спуск и доведите до сходимости (евклидово расстояние между векторами весов на соседних итерациях должно быть не больше 1e-5).**

Рекомендуется ограничить сверху число итераций десятью тысячами.

In [17]:
w_no_reg, iter_no = gradient_descent(X, y, C=0, max_iter=10000)
w_reg, iter_reg = gradient_descent(X, y, C=10, max_iter=10000)

print(f"Веса без регуляризации: w = ({w_no_reg[0]:.8f}, {w_no_reg[1]:.8f})")
print(f"Веса с регуляризацией:  w = ({w_reg[0]:.8f}, {w_reg[1]:.8f})")

Веса без регуляризации: w = (0.28781162, 0.09198330)
Веса с регуляризацией:  w = (0.02855875, 0.02478014)


### **5. Какое значение принимает AUC-ROC на обучении без регуляризации и при ее использовании?**

Эти величины будут ответом на задание. В качестве ответа приведите два числа через пробел. Обратите внимание, что на вход функции roc_auc_score нужно подавать оценки вероятностей, подсчитанные обученным алгоритмом. Для этого воспользуйтесь сигмоидной функцией: a(x) = 1/(1 + exp(-w1x1 - w2x2)).

In [22]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

scores_no = sigmoid(X @ w_no_reg)
scores_reg = sigmoid(X @ w_reg)

auc_no = roc_auc_score(y, scores_no)
auc_reg = roc_auc_score(y, scores_reg)

print(f"  Без регуляризации (C=0):  AUC-ROC = {auc_no:.6f}")
print(f"  С регуляризацией (C=10): AUC-ROC = {auc_reg:.6f}")

answer = f"{auc_no:.3f} {auc_reg:.3f}"

with open('answer.txt', 'w') as f:
    f.write(answer)

  Без регуляризации (C=0):  AUC-ROC = 0.926857
  С регуляризацией (C=10): AUC-ROC = 0.936286


### **6. Попробуйте поменять длину шага.**

In [38]:
#Исследование влияния длины шага (k)
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def gradient_descent_check(X, y, C=0, k=0.1, max_iter=10000, eps=1e-5):
    w = np.zeros(X.shape[1])
    for i in range(max_iter):
        sig = sigmoid(y * np.dot(X, w))
        grad = (1/len(y)) * np.sum(y * X.T * (1 - sig), axis=1) - C * w
        w_new = w + k * grad
        
        # Проверка на расходимость (веса становятся слишком большими)
        if np.max(np.abs(w_new)) > 1e10:
            return False, i + 1, w_new  # Расходится
        
        if np.linalg.norm(w_new - w) < eps:
            return True, i + 1, w_new  # Сошёлся
        
        w = w_new
    
    return False, max_iter, w  # Не сошёлся (достигнут лимит)

# Исследуем разные шаги
k_values = [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 1.5, 2.0, 5.0]

print("\nИсследование влияния шага (k) на сходимость:")
print("-" * 80)
print(f"{'Шаг (k)':<12} {'Итераций':<12} {'Сошёлся?':<12} {'Веса (w1, w2)':<35}")
print("-" * 80)

for k in k_values:
    converged, iters, w = gradient_descent_check(X, y, C=0, k=k, max_iter=10000)
    
    if converged:
        print(f"{k:<12} {iters:<12} {'Да':<12} ({w[0]:.4f}, {w[1]:.4f})")
    else:
        print(f"{k:<12} {iters:<12} {'Нет':<12} ({w[0]:.4f}, {w[1]:.4f})")


Исследование влияния шага (k) на сходимость:
--------------------------------------------------------------------------------
Шаг (k)      Итераций     Сошёлся?     Веса (w1, w2)                      
--------------------------------------------------------------------------------
0.001        5126         Да           (0.2570, 0.1198)
0.01         1479         Да           (0.2850, 0.0946)
0.05         431          Да           (0.2875, 0.0923)
0.1          244          Да           (0.2878, 0.0920)
0.5          60           Да           (0.2881, 0.0918)
1.0          32           Да           (0.2881, 0.0917)
1.5          21           Да           (0.2881, 0.0917)
2.0          10000        Нет          (0.1657, -0.0392)
5.0          10000        Нет          (-0.5890, -0.8682)


Будет ли сходиться алгоритм, если делать более длинные шаги?

Нет, при слишком длинных шагах алгоритм перестаёт сходиться.

При k ≤ 1.5 → алгоритм сходится (веса приближаются к оптимальным)

При k = 2.0 → алгоритм не сходится за 10000 итераций (веса далеки от оптимальных)

При k = 5.0 → алгоритм расходится (веса уходят в отрицательные значения)



Как меняется число итераций при уменьшении длины шага?

Число итераций увеличивается (зависимость обратно пропорциональная).

### **7. Попробуйте менять начальное приближение.**

In [42]:
#Исследование влияния начального приближения
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def gradient_descent_with_init(X, y, w_init, C=0, k=0.1, max_iter=10000, eps=1e-5):
    w = w_init.copy()
    for i in range(max_iter):
        sig = sigmoid(y * np.dot(X, w))
        grad = (1/len(y)) * np.sum(y * X.T * (1 - sig), axis=1) - C * w
        w_new = w + k * grad
        if np.linalg.norm(w_new - w) < eps:
            return w_new, i + 1
        w = w_new
    return w, max_iter

# Исследуем разные начальные приближения
initial_weights = [
    np.array([0.0, 0.0]),      
    np.array([1.0, 1.0]),      
    np.array([-1.0, -1.0]),    
    np.array([10.0, 10.0]),    
    np.array([-10.0, -10.0]),  
    np.array([5.0, -5.0]),     
    np.array([100.0, 100.0]),  
]

print("\nИсследование влияния начального приближения:")
print(f"{'Начальные веса (w1, w2)':<25} {'Итераций':<12} {'Конечные веса (w1, w2)':<35}")

optimal_w = np.array([0.2878, 0.0920])

for w_init in initial_weights:
    w_final, iters = gradient_descent_with_init(X, y, w_init, C=0, k=0.1)
    
    # Оцениваем близость к оптимуму
    distance = np.linalg.norm(w_final - optimal_w)
    
    print(f"({w_init[0]:6.1f}, {w_init[1]:6.1f}){'':<13} "
          f"{iters:<12} "
          f"({w_final[0]:.4f}, {w_final[1]:.4f})  [расст={distance:.4f}]")


Исследование влияния начального приближения:
Начальные веса (w1, w2)   Итераций     Конечные веса (w1, w2)             
(   0.0,    0.0)              244          (0.2878, 0.0920)  [расст=0.0000]
(   1.0,    1.0)              230          (0.2878, 0.0920)  [расст=0.0000]
(  -1.0,   -1.0)              253          (0.2878, 0.0920)  [расст=0.0000]
(  10.0,   10.0)              526          (0.2884, 0.0914)  [расст=0.0008]
( -10.0,  -10.0)              317          (0.2878, 0.0920)  [расст=0.0000]
(   5.0,   -5.0)              532          (0.2884, 0.0914)  [расст=0.0008]
( 100.0,  100.0)              2827         (0.2884, 0.0914)  [расст=0.0008]


C:\Users\Inga\AppData\Local\Temp\ipykernel_23280\1219974656.py:3: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-z))


Влияет ли начальное приближение на что-нибудь?

Нет. При любом начальном приближении (при условии сходимости) алгоритм приходит к одному и тому же оптимальному значению весов (w ≈ 0.288, 0.092).